# 개별종목 조합G — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합G 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합G의 피처 값만 지정합니다.
import json

COMBINATION = 'G'
FEATURE_COLUMNS = (
    'dist_high_60',
    'sma_gap_20_60',
    'relative_ret_5_market',
    'rsi_14',
    'hv_20',
    'turnover_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합G 피처: ('dist_high_60', 'sma_gap_20_60', 'relative_ret_5_market', 'rsi_14', 'hv_20', 'turnover_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,NaN,750,20140217,20140514,0.5012,0.5012,0.0000,0.3055,0.3595,0.0791,0.3905,0.0937,0.1882
1,2,balanced,980,20150123,20150421,0.3835,0.3978,-0.0144,0.3364,0.3533,0.0401,0.3655,0.1898,0.2765
2,3,balanced,1210,20151228,20160328,0.3738,0.3762,-0.0024,0.3546,0.3620,0.0482,0.3748,0.2152,0.2958
3,4,balanced,1439,20161202,20170228,0.4624,0.4617,0.0007,0.3845,0.3948,0.1122,0.4160,0.1893,0.2986
4,5,balanced,1669,20171113,20180207,0.4213,0.3901,0.0312,0.3961,0.4020,0.1095,0.4018,0.3476,0.3858
5,6,balanced,1899,20181024,20190118,0.4139,0.3725,0.0414,0.4059,0.4079,0.1164,0.4230,0.2963,0.3634
6,7,balanced,2129,20190930,20191224,0.4570,0.4781,-0.0212,0.3585,0.3753,0.0842,0.4131,0.2139,0.3108
7,8,balanced,2359,20200902,20201130,0.3908,0.3476,0.0432,0.3883,0.3893,0.0835,0.3967,0.3676,0.3819
8,9,balanced,2589,20210806,20211105,0.3612,0.3914,-0.0302,0.3423,0.3721,0.0523,0.3644,0.1662,0.2563
9,10,balanced,2818,20220714,20221012,0.3937,0.3454,0.0483,0.3838,0.4051,0.1106,0.3892,0.2108,0.3034


,OOS 폴드 평균
accuracy,0.4107
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0138
macro_f1,0.3675
balanced_accuracy,0.3819
mcc,0.0820
pr_auc_macro_ovr,0.3941
down_recall,0.2399
core_harmonic_mean,0.3129


재실행 명령: python scripts/run_stock_model_experiment.py
